# 신약 후보물질 독성 예측 - Day2 모델링

- 목표: `predict_input.csv`(927건)의 독성(`output` 0/1)을 예측한다. 주 지표는 F1(Positive = `label` 1 = 독성 없음), 전부 1 예측 기준선은 0.705다.
- 원칙: 분할을 먼저 하고, 모든 전처리(대체, 상수열 제거, 스케일링)는 `Pipeline` 안에서 학습 데이터로만 fit한다. 교차검증은 Stratified 5-Fold(shuffle).
- 20% 홀드아웃은 최종 모델 확정 전까지 열어 보지 않는다.
- 실험 기록은 `ROADMAP.md` 4절에 옮긴다.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 프로젝트 루트 찾기: 루트나 notebooks/ 어디서 실행해도 data/가 있는 폴더를 루트로 쓴다
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_all  # noqa: E402
from preprocess import CorrFilter, VifFilter, build_pipeline  # noqa: E402

warnings.filterwarnings("ignore")
SEED = 42
TAB = ROOT / "reports/tables"
TAB.mkdir(parents=True, exist_ok=True)

## 1. 데이터, 파생변수, 분할 (먼저 수행, stratify)

`src/data.py`가 데이터를 읽고 RDKit 파생변수를 붙인 뒤 20% 홀드아웃을 분리한다. RDKit 기술자는 행 단위 계산이라 fit이 없어 분할 전에 계산해도 누수가 아니다(`data/cache/`에 저장).

In [2]:
d = load_all(ROOT, seed=SEED)
X, y, X_pred, GROUPS = d.X, d.y, d.X_pred, d.groups
X_tr, X_ho, y_tr, y_ho = d.X_tr, d.X_ho, d.y_tr, d.y_ho
CONT = GROUPS["cont"]

print({k: len(v) for k, v in GROUPS.items()}, "| X", X.shape, "| X_pred", X_pred.shape)
print(f"학습 {X_tr.shape[0]:,} / 홀드아웃 {X_ho.shape[0]:,} | label 1 비율: 학습 {y_tr.mean():.3f}, 홀드아웃 {y_ho.mean():.3f}")

{'cont': 4, 'rdkit': 214, 'fp': 3072} | X (8349, 3290) | X_pred (927, 3290)
학습 6,679 / 홀드아웃 1,670 | label 1 비율: 학습 0.544, 홀드아웃 0.544


## 3. 파이프라인과 교차검증 도구

수치형(연속형 + RDKit)은 중앙값 대체와 상수열 제거를, 핑거프린트는 상수열 제거를 fold 안에서 학습한다.

In [3]:
def build(model, groups, scale=False, corr=None, vif=None):
    return build_pipeline(model, GROUPS, groups, scale=scale, corr=corr, vif=vif)


CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
RESULTS = []


def run(name, model, groups, scale=False, n_jobs=5, corr=None, vif=None):
    pipe = build(model, groups, scale, corr, vif)
    cvr = cross_validate(pipe, X_tr, y_tr, cv=CV, scoring={"f1": "f1", "auc": "roc_auc"}, n_jobs=n_jobs)
    row = {"모델": name, "피처군": "+".join(groups), "상관필터": corr or "-", "VIF필터": vif or "-", "F1": cvr["test_f1"].mean(), "F1 std": cvr["test_f1"].std(),
           "AUC": cvr["test_auc"].mean(), "fit 초": cvr["fit_time"].mean()}
    RESULTS.append(row)
    print(f"{name:<28}{row['피처군']:<16} corr {str(row['상관필터']):<5} vif {str(row['VIF필터']):<4} F1 {row['F1']:.4f} ± {row['F1 std']:.4f} | AUC {row['AUC']:.4f} | {row['fit 초']:.1f}s")
    return row

## 4. 기준선 모델 (5-Fold CV, 학습 80%)

In [4]:
run("Dummy(전부 1)", DummyClassifier(strategy="constant", constant=1), ["cont"])
run("Logistic", LogisticRegression(max_iter=3000), ["cont"], scale=True)
run("Logistic", LogisticRegression(max_iter=3000), ["cont", "rdkit"], scale=True)
run("DecisionTree(depth5)", DecisionTreeClassifier(max_depth=5, random_state=SEED), ["cont", "rdkit"])
run("HistGB", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED), ["cont", "rdkit"])
run("HistGB", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED), ["cont", "rdkit", "fp"])
run("RandomForest(300)", RandomForestClassifier(n_estimators=300, n_jobs=3, random_state=SEED), ["cont", "rdkit"], n_jobs=5)
run("RandomForest(300)", RandomForestClassifier(n_estimators=300, n_jobs=3, random_state=SEED), ["cont", "rdkit", "fp"], n_jobs=5)

Dummy(전부 1)                 cont             corr -     vif -    F1 0.7046 ± 0.0002 | AUC 0.5000 | 0.1s


Logistic                    cont             corr -     vif -    F1 0.6735 ± 0.0145 | AUC 0.6720 | 0.1s


Logistic                    cont+rdkit       corr -     vif -    F1 0.7458 ± 0.0086 | AUC 0.7881 | 0.3s


DecisionTree(depth5)        cont+rdkit       corr -     vif -    F1 0.7129 ± 0.0238 | AUC 0.7306 | 0.3s


HistGB                      cont+rdkit       corr -     vif -    F1 0.8075 ± 0.0062 | AUC 0.8656 | 15.5s


HistGB                      cont+rdkit+fp    corr -     vif -    F1 0.8199 ± 0.0092 | AUC 0.8777 | 51.2s


RandomForest(300)           cont+rdkit       corr -     vif -    F1 0.8104 ± 0.0119 | AUC 0.8657 | 3.3s


RandomForest(300)           cont+rdkit+fp    corr -     vif -    F1 0.8197 ± 0.0078 | AUC 0.8765 | 4.4s


{'모델': 'RandomForest(300)',
 '피처군': 'cont+rdkit+fp',
 '상관필터': '-',
 'VIF필터': '-',
 'F1': np.float64(0.8197060579473912),
 'F1 std': np.float64(0.0077523798902973656),
 'AUC': np.float64(0.8764578234911908),
 'fit 초': np.float64(4.422765827178955)}

In [5]:
res = pd.DataFrame(RESULTS).sort_values("F1", ascending=False)
res.round(4).to_csv(TAB / "modeling_cv_baselines.csv", index=False)
res.round(4)

,모델,피처군,상관필터,VIF필터,F1,F1 std,AUC,fit 초
5,HistGB,cont+rdkit+fp,-,-,0.8199,0.0092,0.8777,51.2378
7,RandomForest(300),cont+rdkit+fp,-,-,0.8197,0.0078,0.8765,4.4228
6,RandomForest(300),cont+rdkit,-,-,0.8104,0.0119,0.8657,3.2504
4,HistGB,cont+rdkit,-,-,0.8075,0.0062,0.8656,15.5492
2,Logistic,cont+rdkit,-,-,0.7458,0.0086,0.7881,0.2720
3,DecisionTree(depth5),cont+rdkit,-,-,0.7129,0.0238,0.7306,0.3401
0,Dummy(전부 1),cont,-,-,0.7046,0.0002,0.5000,0.1004
1,Logistic,cont,-,-,0.6735,0.0145,0.6720,0.0874


## 5. 제출 1: HistGB(cont+rdkit+fp), 전체 train으로 재학습

CV 점수(F1 약 0.82)를 근거로 선택했다. 홀드아웃은 열지 않았다. 임계값은 0.5(튜닝 전)다. 작성자 이름 확정 전이라 파일명은 임시다.

In [6]:
SUB = ROOT / "submissions"
SUB.mkdir(exist_ok=True)

final = build(HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED), ["cont", "rdkit", "fp"])
final.fit(X, y)
proba = final.predict_proba(X_pred)[:, 1]
sub = pd.DataFrame({"SMILES": d.smiles_pred, "output": (proba >= 0.5).astype(int)})
out_path = SUB / "predict_output_작성자.csv"
if not out_path.exists():  # 이미 제출했을 수 있는 파일은 덮어쓰지 않는다
    sub.to_csv(out_path, index=False)
print(sub.shape, "| 예측 label 1 비율", round(sub["output"].mean(), 3), "(학습 데이터는 0.544)")

(927, 2) | 예측 label 1 비율 0.562 (학습 데이터는 0.544)


## 6. 임계값 조정 (OOF 확률로 F1 최대 지점 찾기)

학습 80%의 5-Fold 검증 예측 확률(OOF)로 임계값을 스캔한다. 임계값을 같은 데이터에서 골랐으므로 약간 낙관적이다. 홀드아웃에서 최종 확인한다.

In [7]:
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_predict

THS = np.round(np.arange(0.30, 0.71, 0.01), 2)
OOF = {}


def scan_threshold(name, model, groups, corr=None):
    pipe = build(model, groups, corr=corr)
    proba = cross_val_predict(pipe, X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=5)[:, 1]
    OOF[name] = proba
    f1s = np.array([f1_score(y_tr, (proba >= t).astype(int)) for t in THS])
    best = f1s.argmax()
    print(f"{name:<22} F1@0.5 {f1_score(y_tr, (proba >= 0.5).astype(int)):.4f} | 최대 F1 {f1s[best]:.4f} @ 임계값 {THS[best]:.2f} "
          f"| 0.45~0.55 구간 F1 {f1s[(THS >= 0.45) & (THS <= 0.55)].min():.4f}~{f1s[(THS >= 0.45) & (THS <= 0.55)].max():.4f}")
    return f1s


f1_hgb = scan_threshold("HistGB(cont+rdkit+fp)", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED), ["cont", "rdkit", "fp"])
f1_rf = scan_threshold("RF(cont+rdkit+fp)", RandomForestClassifier(n_estimators=300, n_jobs=3, random_state=SEED), ["cont", "rdkit", "fp"])

HistGB(cont+rdkit+fp)  F1@0.5 0.8199 | 최대 F1 0.8233 @ 임계값 0.39 | 0.45~0.55 구간 F1 0.8116~0.8224


RF(cont+rdkit+fp)      F1@0.5 0.8214 | 최대 F1 0.8241 @ 임계값 0.48 | 0.45~0.55 구간 F1 0.8075~0.8241


## 7. 다중공선성: 찾고 해소하기

- **찾기**: 수치형(원본 4 + RDKit 214)에서 |Spearman| > 0.95 쌍을 세고, VIF(상관행렬 역행렬의 대각)가 10을 넘는 변수를 센다.
- **해소**: `CorrFilter`로 상관이 높은 변수를 하나만 남긴다. 라벨을 쓰지 않아 fold 안에서 fit해도 누수가 없고, 열 순서상 원본 수치형이 먼저 남는다.
- 트리 모델은 다중공선성이 있어도 성능이 비슷할 수 있다. 그래도 변수 관계 해석에는 장애가 되므로 전후 성능을 함께 비교한다.

In [8]:
from scipy.stats import rankdata

num_cols = GROUPS["cont"] + GROUPS["rdkit"]
num_imp = SimpleImputer(strategy="median").fit(X_tr[num_cols])          # 학습 데이터 기준
A = pd.DataFrame(num_imp.transform(X_tr[num_cols]), columns=num_cols)
A = A.loc[:, A.nunique() > 1]                                             # 상수열 제외(학습 데이터 기준)


def collinearity_report(df_, label):
    rho = np.abs(np.nan_to_num(np.corrcoef(rankdata(df_.values, axis=0), rowvar=False)))
    np.fill_diagonal(rho, 0)
    n_pairs = int((np.triu(rho) > 0.95).sum())
    pearson = np.corrcoef(df_.values, rowvar=False)
    vif = np.diag(np.linalg.pinv(pearson))
    print(f"{label:<28} 변수 {df_.shape[1]:>3} | |Spearman|>0.95 쌍 {n_pairs:>3} | VIF>10 {int((vif > 10).sum()):>3}개 | VIF 최대 {vif.max():.0f}")
    return pd.Series(vif, index=df_.columns)


vif_before = collinearity_report(A, "해소 전")
for thr in (0.95, 0.90):
    cf = CorrFilter(thr).fit(A.values)
    collinearity_report(A.iloc[:, cf.keep_], f"해소 후 (|Spearman|<={thr})")
print()
print("해소 전 VIF 상위 8:", vif_before.sort_values(ascending=False).head(8).round(0).to_dict())

해소 전                         변수 207 | |Spearman|>0.95 쌍  76 | VIF>10 125개 | VIF 최대 9137252898648


해소 후 (|Spearman|<=0.95)      변수 179 | |Spearman|>0.95 쌍   0 | VIF>10  91개 | VIF 최대 8996527595577


해소 후 (|Spearman|<=0.9)       변수 168 | |Spearman|>0.95 쌍   0 | VIF>10  77개 | VIF 최대 4187553555938

해소 전 VIF 상위 8: {'rd_NumSaturatedRings': 9137252898648.0, 'rd_NumSaturatedHeterocycles': 5976548100580.0, 'rd_NumAromaticCarbocycles': 4146480110687.0, 'rd_NumSaturatedCarbocycles': 2919894403471.0, 'rd_RingCount': 1849040229282.0, 'rd_NumAromaticRings': 1568345733111.0, 'rd_fr_COO': 172948016719.0, 'rd_fr_Al_COO': 135419605865.0}


In [9]:
# 상관 필터가 성능에 주는 영향(fold 안에서 fit)
HGB = lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED)
for thr in (None, 0.95, 0.90):
    run("HistGB", HGB(), ["cont", "rdkit"], corr=thr)
for thr in (None, 0.95):
    run("HistGB", HGB(), ["cont", "rdkit", "fp"], corr=thr)

HistGB                      cont+rdkit       corr -     vif -    F1 0.8075 ± 0.0062 | AUC 0.8656 | 7.6s


HistGB                      cont+rdkit       corr 0.95  vif -    F1 0.8094 ± 0.0100 | AUC 0.8651 | 16.7s


HistGB                      cont+rdkit       corr 0.9   vif -    F1 0.8056 ± 0.0089 | AUC 0.8672 | 8.1s


HistGB                      cont+rdkit+fp    corr -     vif -    F1 0.8199 ± 0.0092 | AUC 0.8777 | 54.3s


HistGB                      cont+rdkit+fp    corr 0.95  vif -    F1 0.8158 ± 0.0075 | AUC 0.8777 | 55.3s


### 7-2. VIF 필터로 선형 결합형 다중공선성까지 해소

상관 필터(쌍 단위)로는 `RingCount = 방향족 고리 + 포화 고리`처럼 여러 변수의 합으로 생기는 중복이 남는다. VIF가 10을 넘는 변수를 가장 큰 것부터 하나씩 제거한다(`VifFilter`, 라벨 미사용). 해석이 쉬운 원본 수치형 `clogp`, `qed`, `sa_score`는 보호하고 RDKit 파생변수를 먼저 제거한다. `MolWt`는 RDKit 질량 기술자(`HeavyAtomMolWt`, `ExactMolWt` 등)와 구조적으로 중복이라 보호하지 않는다. 보호한 변수는 VIF가 커도 남기므로 그 값은 따로 보고한다.

In [10]:
vf = VifFilter(10, protect=3).fit(A.values)
A_vif = A.iloc[:, vf.keep_]
vif_after = collinearity_report(A_vif, "해소 후 (VIF<=10, 원본3 보호)")
print("VIF>10으로 남은 변수(보호한 원본):", vif_after[vif_after > 10].round(1).to_dict(), "| 나머지 최대", round(vif_after[vif_after <= 10].max(), 2))
dropped = [c for c in A.columns if c not in A_vif.columns]
print(f"제거 {len(dropped)}개, 유지 {A_vif.shape[1]}개 | 원본 수치형 유지 여부:", {c: (c in A_vif.columns) for c in CONT})
print("제거된 변수 예:", dropped[:12])

해소 후 (VIF<=10, 원본3 보호)       변수 134 | |Spearman|>0.95 쌍   0 | VIF>10   1개 | VIF 최대 104
VIF>10으로 남은 변수(보호한 원본): {'clogp': 103.8} | 나머지 최대 8.99
제거 73개, 유지 134개 | 원본 수치형 유지 여부: {'clogp': True, 'qed': True, 'sa_score': True, 'MolWt': False}
제거된 변수 예: ['MolWt', 'rd_MaxEStateIndex', 'rd_SPS', 'rd_HeavyAtomMolWt', 'rd_ExactMolWt', 'rd_NumValenceElectrons', 'rd_MaxPartialCharge', 'rd_MaxAbsPartialCharge', 'rd_FpDensityMorgan1', 'rd_FpDensityMorgan2', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_CHGLO']


In [11]:
# 해소 전후 성능: 선형 모델은 영향이 크고 트리 모델은 작을 것으로 기대한다
run("Logistic", LogisticRegression(max_iter=3000), ["cont", "rdkit"], scale=True)
run("Logistic", LogisticRegression(max_iter=3000), ["cont", "rdkit"], scale=True, vif=10)
run("HistGB", HGB(), ["cont", "rdkit"], vif=10)
run("HistGB", HGB(), ["cont", "rdkit", "fp"], vif=10)

Logistic                    cont+rdkit       corr -     vif -    F1 0.7458 ± 0.0086 | AUC 0.7881 | 0.3s


Logistic                    cont+rdkit       corr -     vif 10   F1 0.7362 ± 0.0103 | AUC 0.7745 | 0.4s


HistGB                      cont+rdkit       corr -     vif 10   F1 0.8047 ± 0.0085 | AUC 0.8631 | 6.6s


HistGB                      cont+rdkit+fp    corr -     vif 10   F1 0.8188 ± 0.0062 | AUC 0.8783 | 54.8s


{'모델': 'HistGB',
 '피처군': 'cont+rdkit+fp',
 '상관필터': '-',
 'VIF필터': 10,
 'F1': np.float64(0.8187894350602027),
 'F1 std': np.float64(0.006220052027892814),
 'AUC': np.float64(0.8783195577776611),
 'fit 초': np.float64(54.783442974090576)}